# 03a — Tokenizer & Data Prep
**Run once. Takes ~5–10 minutes. Must complete before running 03b.**

### What this does
1. Loads all synthetic `.jsonl` files
2. Trains a shared SentencePiece BPE tokenizer on the combined corpus
3. Tokenizes and caches ALL datasets to disk as `.pt` tensors
4. Tokenizes FLORES dev + devtest and caches them too

### Output (saved to `notebooks/models/`)
- `shared_spm.model` — SentencePiece tokenizer
- `cache_beam_M1.pt`, `cache_beam_M10.pt`, ... — tokenized training data
- `cache_flores_dev.pt`, `cache_flores_devtest.pt` — tokenized eval data
- `vocab_info.json` — vocab size, special token IDs

### After this notebook
→ Open `03b_train.ipynb` to train the student model

In [1]:
# Install if needed
# !pip install --quiet sentencepiece torch tqdm

In [2]:
import json
import os
import random
import time
from pathlib import Path
from typing import List, Tuple, Optional

import torch
import sentencepiece as spm
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)

# ── KAGGLE-SAFE PATHS ─────────────────────────────────────────────────────
# Inputs may live in /kaggle/input/<your-dataset>/data/...
# Outputs/checkpoints should live in /kaggle/working so they can be saved.
IS_KAGGLE = Path("/kaggle/working").exists()
WORK_ROOT = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()

def _candidate_roots():
    roots = [WORK_ROOT, Path.cwd(), Path.cwd().parent]
    if Path("/kaggle/input").exists():
        for d in Path("/kaggle/input").iterdir():
            if d.is_dir():
                roots += [d, d / "MHD2", d / "mhd2"]
    return list(dict.fromkeys([r.resolve() for r in roots if r.exists()]))

def find_dir(rel_path: str) -> Path:
    rel = Path(rel_path)
    for root in _candidate_roots():
        p = root / rel
        if p.exists():
            return p
    searched = "\n".join(str(root / rel) for root in _candidate_roots())
    raise FileNotFoundError(f"Could not find {rel_path}. Searched:\n{searched}")

SYNTH_DIR   = find_dir("data/synthetic")
FLORES_DIR  = find_dir("data/flores")
FLORES_DEV  = FLORES_DIR / "dev.jsonl"
FLORES_TEST = FLORES_DIR / "devtest.jsonl"

MODEL_DIR   = WORK_ROOT / "notebooks" / "models"
CACHE_DIR   = MODEL_DIR / "cache"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SPM_PREFIX  = str(MODEL_DIR / "shared_spm")
SPM_MODEL   = SPM_PREFIX + ".model"
VOCAB_INFO  = MODEL_DIR / "vocab_info.json"

# ── SETTINGS ──────────────────────────────────────────────────────────────
VOCAB_SIZE  = 32_000
MAX_LENGTH  = 128     # tokens including BOS/EOS

DATASETS = {
    "beam_M1":   "eng_swh_beam_M1.jsonl",
    "beam_M10":  "eng_swh_beam_M10.jsonl",
    "top_p_M10": "eng_swh_top_p_M10.jsonl",
    "top_k_M10": "eng_swh_top_k_M10.jsonl",
    "dbs_M10":   "eng_swh_dbs_M10.jsonl",
    "mbr_M10":   "eng_swh_mbr_M10.jsonl",
}

print("✓ Paths configured")
print(f"  Kaggle mode:    {IS_KAGGLE}")
print(f"  Synthetic dir:  {SYNTH_DIR}")
print(f"  FLORES dir:     {FLORES_DIR}")
print(f"  Output models:  {MODEL_DIR}")
print(f"  Cache dir:      {CACHE_DIR}")


✓ Paths configured
  Kaggle mode:    False
  Synthetic dir:  C:\Users\nirmi\Desktop\MHD2\data\synthetic
  FLORES dir:     C:\Users\nirmi\Desktop\MHD2\data\flores
  Output models:  c:\Users\nirmi\Desktop\MHD2\notebooks\notebooks\models
  Cache dir:      c:\Users\nirmi\Desktop\MHD2\notebooks\notebooks\models\cache


c:\Users\nirmi\Desktop\MHD2\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ── STEP 1: Load synthetic/FLORES data ────────────────────────────────────

def load_jsonl(path: Path, expand_hypotheses: bool = True) -> Tuple[List[str], List[str]]:
    """
    Load synthetic .jsonl.
    Expected fields are usually src/tgt; this also accepts source/reference.
    expand_hypotheses=True: all M hypotheses become separate training pairs.
    expand_hypotheses=False: only hyp_id==0 is kept.
    """
    srcs, tgts = [], []
    if not path.exists():
        raise FileNotFoundError(f"Missing JSONL file: {path}")

    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            ex = json.loads(line)

            if not expand_hypotheses and ex.get("hyp_id", 0) != 0:
                continue

            s = ex.get("src", ex.get("source", ""))
            t = ex.get("tgt", ex.get("target", ex.get("reference", "")))

            s, t = str(s).strip(), str(t).strip()
            if s and t:
                srcs.append(s)
                tgts.append(t)

    if not srcs:
        raise ValueError(f"No usable src/tgt pairs found in {path}")
    return srcs, tgts


def load_flores(path: Path) -> Tuple[List[str], List[str]]:
    """
    Load FLORES-style JSONL.
    Accepts either source/reference or src/ref fields.
    """
    srcs, refs = [], []
    if not path.exists():
        raise FileNotFoundError(f"Missing FLORES file: {path}")

    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            ex = json.loads(line)
            src = ex.get("source", ex.get("src", ""))
            ref = ex.get("reference", ex.get("ref", ex.get("tgt", "")))
            src, ref = str(src).strip(), str(ref).strip()
            if src and ref:
                srcs.append(src)
                refs.append(ref)

    if not srcs:
        raise ValueError(f"No usable FLORES rows found in {path}")
    return srcs, refs


# Use beam_M10 for tokenizer if available; otherwise fall back to all available synthetic files.
print("Loading data for tokenizer training...")
preferred = SYNTH_DIR / DATASETS["beam_M10"]
tokenizer_files = [preferred] if preferred.exists() else sorted(SYNTH_DIR.glob("*.jsonl"))

all_src, all_tgt = [], []
for fp in tokenizer_files:
    srcs, tgts = load_jsonl(fp, expand_hypotheses=False)
    all_src.extend(srcs)
    all_tgt.extend(tgts)
    print(f"  + {fp.name}: {len(srcs)} pairs")

print(f"  Tokenizer corpus: {len(all_src)} src + {len(all_tgt)} tgt = {len(all_src)+len(all_tgt)} lines")


Loading data for tokenizer training...
  + eng_swh_beam_M10.jsonl: 10000 pairs
  Tokenizer corpus: 10000 src + 10000 tgt = 20000 lines


In [4]:
# ── STEP 2: Train SentencePiece tokenizer ─────────────────────────────────

if Path(SPM_MODEL).exists():
    print(f"✓ Tokenizer already exists: {SPM_MODEL} — skipping training")
else:
    CORPUS_FILE = str(CACHE_DIR / "spm_corpus.txt")
    print(f"Writing corpus to {CORPUS_FILE}...")
    with open(CORPUS_FILE, "w", encoding="utf-8") as f:
        for s in all_src:
            f.write(s + "\n")
        for t in all_tgt:
            f.write(t + "\n")

    print(f"Training SentencePiece BPE (requested vocab={VOCAB_SIZE})...")
    print("Note: SentencePiece does not show a percentage bar; INFO logs mean it is running.")
    start = time.time()
    spm.SentencePieceTrainer.train(
        input=CORPUS_FILE,
        model_prefix=SPM_PREFIX,
        vocab_size=VOCAB_SIZE,
        model_type="bpe",
        character_coverage=1.0,
        pad_id=0, unk_id=1, bos_id=2, eos_id=3,
        pad_piece="<pad>", unk_piece="<unk>",
        bos_piece="<s>", eos_piece="</s>",
        shuffle_input_sentence=True,
        num_threads=max(1, os.cpu_count() or 4),
        hard_vocab_limit=False,   # prevents failure when corpus is too small for 32k vocab
    )
    print(f"✓ Tokenizer saved: {SPM_MODEL} in {(time.time()-start)/60:.1f} min")

# Load tokenizer
sp = spm.SentencePieceProcessor(model_file=SPM_MODEL)
PAD_ID, UNK_ID, BOS_ID, EOS_ID = sp.pad_id(), sp.unk_id(), sp.bos_id(), sp.eos_id()
ACTUAL_VOCAB = sp.get_piece_size()

print(f"\nVocab size: {ACTUAL_VOCAB} | PAD={PAD_ID} UNK={UNK_ID} BOS={BOS_ID} EOS={EOS_ID}")

# Save vocab info for downstream notebooks.
# Important: save the Kaggle /kaggle/working path, so 03b/03c can load it after copying outputs.
vocab_info = {
    "vocab_size": ACTUAL_VOCAB,
    "pad_id": PAD_ID, "unk_id": UNK_ID,
    "bos_id": BOS_ID, "eos_id": EOS_ID,
    "max_length": MAX_LENGTH,
    "spm_model": str(MODEL_DIR / "shared_spm.model"),
}
with open(VOCAB_INFO, "w", encoding="utf-8") as f:
    json.dump(vocab_info, f, indent=2)
print(f"✓ Vocab info saved: {VOCAB_INFO}")


Writing corpus to c:\Users\nirmi\Desktop\MHD2\notebooks\notebooks\models\cache\spm_corpus.txt...
Training SentencePiece BPE (requested vocab=32000)...
Note: SentencePiece does not show a percentage bar; INFO logs mean it is running.
✓ Tokenizer saved: c:\Users\nirmi\Desktop\MHD2\notebooks\notebooks\models\shared_spm.model in 0.1 min

Vocab size: 32000 | PAD=0 UNK=1 BOS=2 EOS=3
✓ Vocab info saved: c:\Users\nirmi\Desktop\MHD2\notebooks\notebooks\models\vocab_info.json


In [5]:
# ── STEP 3: Roundtrip + UNK sanity checks ─────────────────────────────────

print("Roundtrip check:")
for s in ["Hello, world!", "Habari yako?", all_src[0][:60], all_tgt[0][:60]]:
    ids = sp.encode(s, add_bos=False, add_eos=False)
    dec = sp.decode(ids)
    ok = "✅" if dec.strip().lower() == s.strip().lower() else "⚠️"
    print(f"  {ok}  '{s[:45]}'  →  {len(ids)} tokens  →  '{dec[:45]}'")

# UNK rate on M1 corpus
unk_hits = sum(1 for s in all_src[:1000] for tid in sp.encode(s) if tid == UNK_ID)
total_tok = sum(len(sp.encode(s)) for s in all_src[:1000])
print(f"\nUNK rate on 1k src sentences: {100*unk_hits/total_tok:.2f}% (want < 1%)")

Roundtrip check:
  ✅  'Hello, world!'  →  4 tokens  →  'Hello, world!'
  ✅  'Habari yako?'  →  3 tokens  →  'Habari yako?'
  ✅  'YES! I’ve actually gone without shampoo and s'  →  14 tokens  →  'YES! I’ve actually gone without shampoo and s'
  ✅  'Kwa kweli nimekuwa bila sampuli na sabuni kwa'  →  10 tokens  →  'Kwa kweli nimekuwa bila sampuli na sabuni kwa'

UNK rate on 1k src sentences: 0.00% (want < 1%)


In [6]:
# ── STEP 4: Tokenize and cache ALL datasets ────────────────────────────────

def encode_with_specials(text: str) -> torch.Tensor:
    # Keep EOS even after truncation.
    ids = [BOS_ID] + sp.encode(text, add_bos=False, add_eos=False)[:MAX_LENGTH - 2] + [EOS_ID]
    return torch.tensor(ids, dtype=torch.long)


def tokenize_and_cache(srcs: List[str], tgts: List[str], cache_path: Path, desc: str):
    """
    Tokenize parallel pairs and save as list of dicts.
    Each dict: {src: LongTensor, tgt: LongTensor}
    """
    if cache_path.exists():
        print(f"  ✓ Already cached: {cache_path.name} — skipping")
        return

    data = []
    skipped = 0
    for src, tgt in tqdm(list(zip(srcs, tgts)), total=len(srcs), desc=desc):
        src_ids = encode_with_specials(src)
        tgt_ids = encode_with_specials(tgt)

        if len(src_ids) < 3 or len(tgt_ids) < 3:
            skipped += 1
            continue

        data.append({"src": src_ids, "tgt": tgt_ids})

    torch.save(data, cache_path)
    print(f"  ✓ Cached {len(data)} pairs → {cache_path.name}  (skipped {skipped})")


# Cache all synthetic datasets
print("Tokenizing synthetic datasets...")
cached_any = False
for key, fname in DATASETS.items():
    fpath = SYNTH_DIR / fname
    if not fpath.exists():
        print(f"  ⚠️  Not found: {fname} — skipping")
        continue
    cache_path = CACHE_DIR / f"cache_{key}.pt"
    srcs, tgts = load_jsonl(fpath, expand_hypotheses=True)
    tokenize_and_cache(srcs, tgts, cache_path, desc=key)
    cached_any = True

if not cached_any:
    raise FileNotFoundError(f"No expected synthetic JSONL files found in {SYNTH_DIR}")

# Cache FLORES eval sets
print("\nTokenizing FLORES eval sets...")
for split_name, fpath in [("flores_dev", FLORES_DEV), ("flores_devtest", FLORES_TEST)]:
    cache_path = CACHE_DIR / f"cache_{split_name}.pt"
    srcs, refs = load_flores(fpath)

    raw_path = CACHE_DIR / f"raw_{split_name}.json"
    with open(raw_path, "w", encoding="utf-8") as f:
        json.dump({"src": srcs, "ref": refs}, f, ensure_ascii=False, indent=2)

    tokenize_and_cache(srcs, refs, cache_path, desc=split_name)
    print(f"  ✓ Raw text saved: {raw_path.name}")

print("\n🎉 All caching complete!")


Tokenizing synthetic datasets...


beam_M1: 100%|██████████| 10000/10000 [00:01<00:00, 9830.74it/s]


  ✓ Cached 10000 pairs → cache_beam_M1.pt  (skipped 0)


beam_M10: 100%|██████████| 100000/100000 [00:08<00:00, 11590.44it/s]


  ✓ Cached 100000 pairs → cache_beam_M10.pt  (skipped 0)


...

  ✓ Cached 100000 pairs → cache_mbr_M10.pt  (skipped 0)

Tokenizing FLORES eval sets...


flores_dev: 100%|██████████| 997/997 [00:00<00:00, 9764.38it/s]


  ✓ Cached 997 pairs → cache_flores_dev.pt  (skipped 0)
  ✓ Raw text saved: raw_flores_dev.json


flores_devtest: 100%|██████████| 1012/1012 [00:00<00:00, 9822.36it/s]


  ✓ Cached 1012 pairs → cache_flores_devtest.pt  (skipped 0)
  ✓ Raw text saved: raw_flores_devtest.json

🎉 All caching complete!


In [7]:
# ── STEP 5: Final verification ────────────────────────────────────────────

print("Cache summary:")
for f in sorted(CACHE_DIR.glob("*.pt")):
    data = torch.load(f, weights_only=False)
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:<35}  {len(data):>7} pairs  {size_mb:5.1f} MB")

print("\nVocab info:")
print(json.dumps(vocab_info, indent=2))

print("\n✅ 03a complete. You can now run 03b_train.ipynb")

Cache summary:
  cache_beam_M1.pt                       10000 pairs   10.1 MB
  cache_beam_M10.pt                     100000 pairs  101.9 MB
  cache_dbs_M10.pt                      100000 pairs  102.3 MB
  cache_flores_dev.pt                      997 pairs    1.0 MB
  cache_flores_devtest.pt                 1012 pairs    1.0 MB
  cache_mbr_M10.pt                      100000 pairs  101.4 MB
  cache_top_k_M10.pt                    100000 pairs  102.9 MB
  cache_top_p_M10.pt                    100000 pairs  102.1 MB

Vocab info:
{
  "vocab_size": 32000,
  "pad_id": 0,
  "unk_id": 1,
  "bos_id": 2,
  "eos_id": 3,
  "max_length": 128,
  "spm_model": "c:\\Users\\nirmi\\Desktop\\MHD2\\notebooks\\notebooks\\models\\shared_spm.model"
}

✅ 03a complete. You can now run 03b_train.ipynb
